# Piece 2 - joint sentence + token head

A second output head on the shared BiLSTM predicts whether the whole tweet is
offensive:

    loss = token_loss + lambda * sentence_loss

Lambda is swept over 0, 0.1, 0.3, 0.5, 1.0. Lambda 0 builds the model with no
sentence head, so it is the control for this sweep.

Scored on validation over 5 seeds. Test is not scored here.

## How to run

1. Runtime -> Change runtime type -> T4 GPU
2. Set `OWNER` in the Drive cell
3. Run top to bottom

Colab stops when the tab closes, so the tab has to stay open for the run.

## GPU

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'

## Install

In [ ]:
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
print('ready')

## Code

In [ ]:
REPO = 'https://github.com/hatheem-r/project_DNN.git'

import os, shutil
if os.path.exists('/content/project'): shutil.rmtree('/content/project')
%cd /content
!git clone -q $REPO project
%cd /content/project

## Tests

The alignment tests check that piece lists line up with words. A break there
shifts labels against tokens without raising an error.

In [ ]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2

## Drive

Results are copied here so they survive the session ending.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/sold_results'
import os
os.makedirs(OUT, exist_ok=True)

OWNER = 'YOUR_NAME'        # <-- EDIT

print('saving to', OUT, '| owner', OWNER)

## fastText vectors

About 460 MB.

In [ ]:
!mkdir -p embeddings results artifacts
![ -f embeddings/cc.si.300.vec.gz ] || wget -q --show-progress -O embeddings/cc.si.300.vec.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
!ls -lh embeddings/

## Run

Sweeps lambda over 0, 0.1, 0.3, 0.5, 1.0 at 5 seeds each. `--loss` is left at
its default, `cross_entropy`. Roughly 2.5 hours on a T4.

In [ ]:
!python notebooks/09_pieces_234.py --piece2 --owner $OWNER --seeds 1 2 3 4 5 2>&1 | tee results/piece2_report.txt | tail -40
!cp results/results_piece2.csv results/piece2_report.txt $OUT/ && ls -la $OUT/

## Report

From the ranking table at the end of the output:

- lambda = 0 (control): F1 and std
- best lambda, its F1, std, precision and recall
- the verdict line comparing the best lambda against lambda = 0
- the lambda value for Piece 4

The two files are in Drive and in `results/`. To commit:

```
git add results/results_piece2.csv results/piece2_report.txt
git commit -m "piece2 results"
git pull --rebase
git push
```